# Step By Step 03 - Factor Design
----------

Factor design requires factor transformations. **alpha-mind** uses **calc-flow-python** to compose transformations efficiently.

## 1. Fetching Basic Factors

In [ ]:
import asyncio
import datetime as dt


In [ ]:
import numpy as np


In [ ]:
import pyarrow as pa
import pyarrow.compute as pc


## 2. Arithmetic Operations

In [ ]:
rng = np.random.default_rng(42)
n_dates = 6


In [ ]:
# Supports intuitive addition, subtraction, multiplication, and division
n_assets = 12


In [ ]:
# Expressions can be arbitrarily long
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


## 3. Window-Based Rolling Calculations

Financial calculations inevitably involve time-series processing, which in turn requires rolling operations. **calc-flow-python** provides comprehensive support for these operations.

In [ ]:
# A standard rolling mean
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


## 4. Cross-Sectional Processing

Financial factor analysis often compares individual stocks. This requires cross-sectional processing, such as ranking CSI 300 constituents by an indicator:

In [ ]:
# This example uses CSI 300 constituents
from alphamind.data.transformer import Transformer


In [ ]:
from alphamind.formula import LAST, MA, SHIFT, CSQuantiles, CSRank, decode_formula, encode_formula

expressions = {
    "raw": LAST("factor"),
    "lag": SHIFT("factor", 1),
    "mean": MA(3, "factor"),
    "rank": CSRank("factor"),
    "percentile": CSQuantiles("factor"),
    "combined": LAST("factor") + 0.5 * LAST("value"),
}


## 5. Complete Composition

All the operations above can be composed with one another.

In [ ]:
result = await asyncio.to_thread(Transformer(expressions).transform, "code", panel)
restored = decode_formula(encode_formula(expressions["lag"]))


In [ ]:
restored_result = await asyncio.to_thread(Transformer({"lag": restored}).transform, "code", panel)


In [ ]:
assert result["lag"].to_pylist() == restored_result["lag"].to_pylist()
result
